# CSE 25 - Introduction to Artificial Intelligence
## Week 5 Tuesday: Logistic regression and forward computation

**Learning Objectives:**
- Define the sigmoid function
- Explain how the sigmoid function is used to describe confidence in classification predictions
- Define binary cross-entropy
- Explain why binary cross-entropy works as a loss function for logistic regression
- Define logistic regression
- Explain the role of a loss function in learning
- Interpret a computation graph for a simple model  
- Describe the forward computation from inputs to loss  

Instructions

Use your copy of this notebook on Datahub and complete it during class. Work through the cells below **in order**. You may discuss with your neighbors, but make sure you understand each step yourself.


SUBMISSION:
When finished, download it as a PDF and upload it to Gradescope under `In-Class – Week 5 Tuesday` to receive credit. To download it as a PDF, on DataHub go to `File -> Save and Export Notebook As -> PDF`.

## Confidence instead of hard thresholds

With perceptrons, for each input example we predicted a +1 / -1 label. Moreover, the total perceptron loss function $\\mathcal = \\sum_j \\max(0, −y_jz_j)$  is zero when all training data are classified correctly, without capturing anything about whether any data points are close to the boundary or not

Sometimes, we don't want a *hard* yes/no.

Instead, we might want to say things like:
- "I'm very confident this is class 1"
- "I'm somewhat confident this is class 1"
- "I'm unsure"
- "I'm somewhat confident this is class -1"
- "I'm confident this is class -1"


We will use ideas from **probability theory**, hence using values between $0$ and $1$.

To capture the **confidence** we have when using the score to classify an example, we apply a function to the score that \"squishes\" it to a value between 0 and 1. One common function used for this is called the **sigmoid** function, defined as

$$
\sigma(z) = \frac{1}{1 + e^{-z}}
$$


In [ ]:
import math
# Complete the sigmoid function below.
def sigmoid(z):
    """
    z: a numeric value.
    Return sigmoid(z) = 1 / (1 + exp(-z)).
    """
    # YOUR CODE HERE
    # You can use math.exp() to compute the exponential of a number.

    raise NotImplementedError

In [ ]:
# Use assertions for 4 key test cases
assert abs(sigmoid(0) - 0.5) < 1e-7, "sigmoid(0) failed"
assert abs(sigmoid(1) - 0.7310586) < 1e-7, "sigmoid(1) failed"
assert abs(sigmoid(-1) - 0.2689414) < 1e-7, "sigmoid(-1) failed"
assert abs(sigmoid(-10) - 0.0000454) < 1e-7, "sigmoid(-10) failed"
assert abs(sigmoid(10) - 0.9999546) < 1e-7, "sigmoid(10) failed"

print("All key sigmoid test cases passed!")


Q: Why are we not using = in the assert statements above?

`YOUR ANSWER HERE`

Q. What happens if we try to compute `sigmoid(-800)` with the sigmoid implementation above? Why?

`YOUR ANSWER HERE`

In [ ]:
# sigmoid(-800)

The sigmoid function can be written in two equivalent forms:

$$
\sigma(z) = \frac{1}{1 + e^{-z}} = \frac{e^{z}}{1 + e^{z}}
$$

*How?* Multiply by a convenient form of $1$, namely by $\frac{e^z}{e^z}$. Then notice that $e^{-z} e^{z} = 1$.

Both formulas produce the same output for any value of $z$, but we choose which one to use in our code to avoid computation overflow.

In particular, if we are calculating $\sigma(z)$ for a very large number, we use the original form (because $e^{-z}$ will be close to $0$ whereas $e^{z}$ would be too large); if we are calculating $\sigma(z)$ for a very large negative number, we use the second form (because $e^{z}$ will then be close to $0$ whereas $e^{-z}$ would be too large). 

In [ ]:
# Complete the sigmoid function below.
def stable_sigmoid(z):
    """
    This implementation avoids overflow issues by handling large positive and negative values of z separately.

    z: a numeric value.

    if z >= 0: use the 'standard' formula: 1/(1 + exp(-z))
    if z < 0: use the alternative formula to avoid overflow: exp(z) / (1 + exp(z))
    """
    if z >= 0:
        return 1 / (1 + math.exp(-z))
    else:
        ez = math.exp(z)
        return ez / (1 + ez)

In [ ]:
stable_sigmoid(-800)

Now, let's plot the sigmoid function and compare it to the step function we used as the perceptron activation function before.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

z_vals = np.linspace(-10, 10, 400)
step = (z_vals > 0).astype(int)

# Plotting sigmoid function
plt.figure(figsize=(7, 4))
sigmoid_values = [stable_sigmoid(z) for z in z_vals]
plt.plot(z_vals, sigmoid_values, label="sigmoid(z)")
plt.axvline(x=0, color='black', linestyle='-', linewidth=2, alpha=0.7, label="z=0")
plt.xlabel("$z$")
plt.ylabel("sigmoid(z)")
plt.title("Sigmoid Function")
plt.grid(True)
plt.legend()
plt.show()

# Plotting step function
plt.figure(figsize=(7, 4))
plt.plot(z_vals, step, label="step(z)")
plt.axvline(x=0, color='black', linestyle='-', linewidth=2, alpha=0.7, label="z=0")
plt.xlabel("Score (z)")
plt.ylabel("step(z)")
plt.title("Step Function")
plt.grid(True)
plt.ylim(-0.1, 1.1)
plt.legend()
plt.show()

### Logistic regression

Logistic regression means determining the model parameters that minimize an error on the training examples. 

The model means using the sigmoid function as the activation function in a binary classifier:

1. Compute a score: 
$$
z = \sum_{i=1}^n w_i x_i + b
$$

2. Turn the score into a confidence value $p$: $$p = \sigma(z)$$

We have not changed how the score is computed -  we only changed what we do *after* the score.

- To make a prediction:
    - If $p > 0.5$, predict class 1
    - If $p < 0.5$, predict class 0

This means the model outputs a **confidence** (between 0 and 1), and we use a **threshold** (usually 0.5) to decide the final class.

## What impacts the quality of classifiers?

The interactive graph below shows how predictions and evaluation metrics depend on the **decision threshold**, the **data distribution**, and how well the model separates the classes.

**Controls**

- **Threshold**: changes how strict the classifier is.  
  - Lower threshold -> more positives (higher recall, more FP).  
  - Higher threshold -> fewer positives (higher precision, more FN).
- **Positive rate (prevalence)**: changes how common positives are, illustrating class imbalance.
- **Separability**: controls how easy the classification task is (how well scores separate the classes).

**What you are seeing**

- Each example has a **model score** (a confidence value between 0 and 1).
- The histogram shows the distribution of many examples; each bar represents multiple data points.
- The vertical line is the current **threshold**.  
- Scores to the right of the line are predicted as positive.

**Color Modes**

- **By true label**: Colors show the actual class (positive vs negative).
- **By outcome (TP/FP/FN/TN)**: Colors show whether predictions are correct or incorrect:
  - TP and TN are correct predictions.
  - FP are false alarms.
  - FN are missed positives.

**Confusion Matrix**

- **TP**: correct positive predictions  
- **FP**: false alarms  
- **FN**: missed positives  
- **TN**: correct negative predictions  

As you adjust the controls, examples move between these categories, which changes the metrics.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

def confusion_counts(y_true, y_pred):
    y_true_ = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    return tp, fp, fn, tn

def safe_div(num, den):
    return float(num) / float(den) if den != 0 else np.nan

def compute_metrics(tp, fp, fn, tn):
    acc  = safe_div(tp + tn, tp + fp + fn + tn)
    prec = safe_div(tp, tp + fp)
    rec  = safe_div(tp, tp + fn)   # Recall / TPR
    f1   = safe_div(2*prec*rec, prec + rec) if (prec == prec and rec == rec and (prec + rec) != 0) else np.nan
    return acc, prec, rec, f1

def fmt(x):
    return "undefined" if (x != x) else f"{x:.3f}"

def make_dataset(n=100, prevalence=0.2, separability=2.0):
    rng = np.random.default_rng(0)  # fixed seed
    n_pos = int(round(n * prevalence))
    n_neg = n - n_pos

    pos_raw = rng.normal(loc=+separability/2, scale=1.0, size=n_pos)
    neg_raw = rng.normal(loc=-separability/2, scale=1.0, size=n_neg)

    sigmoid = lambda z: 1 / (1 + np.exp(-z))
    pos_scores = sigmoid(pos_raw)
    neg_scores = sigmoid(neg_raw)

    y_true  = np.array([1]*n_pos + [0]*n_neg)
    y_score = np.concatenate([pos_scores, neg_scores])

    idx = rng.permutation(n)
    return y_true[idx], y_score[idx]


def plot_confusion(ax, tp, fp, fn, tn):
    cm = np.array([[tp, fn],
                   [fp, tn]], dtype=float)

    vmax = max(cm.max(), 1.0)
    im = ax.imshow(cm, vmin=0, vmax=vmax, cmap="Blues")

    ax.set_title("Confusion Matrix")
    ax.set_xticks([0, 1], labels=["Pred 1", "Pred 0"])
    ax.set_yticks([0, 1], labels=["Actual 1", "Actual 0"])

    threshold = vmax * 0.55
    for (i, j), v in np.ndenumerate(cm):
        color = "white" if v >= threshold else "black"
        ax.text(j, i, f"{int(v)}", ha="center", va="center",
                color=color, fontsize=12, fontweight="bold")

    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label("Count", rotation=270, labelpad=12)

def plot_score_hist(ax, y_true, y_score, y_pred, thr, color_mode):
    bins = np.linspace(0, 1, 21)

    if color_mode == "By true label":
        pos = y_score[y_true == 1]
        neg = y_score[y_true == 0]
        ax.hist(neg, bins=bins, alpha=0.6, label="Actual 0 (neg)")
        ax.hist(pos, bins=bins, alpha=0.6, label="Actual 1 (pos)")

    elif color_mode == "By outcome (TP/FP/FN/TN)":
        tp_mask = (y_true == 1) & (y_pred == 1)
        fp_mask = (y_true == 0) & (y_pred == 1)
        fn_mask = (y_true == 1) & (y_pred == 0)
        tn_mask = (y_true == 0) & (y_pred == 0)

        ax.hist(y_score[tp_mask], bins=bins, alpha=0.7, label="TP (correct +)")
        ax.hist(y_score[tn_mask], bins=bins, alpha=0.7, label="TN (correct -)")
        ax.hist(y_score[fp_mask], bins=bins, alpha=0.7, label="FP (false alarm)")
        ax.hist(y_score[fn_mask], bins=bins, alpha=0.7, label="FN (missed +)")

    ax.axvline(thr, linewidth=2, label=f"threshold = {thr:.2f}")
    ax.set_xlim(0, 1)
    ax.set_title("Score Distributions + Threshold")
    ax.set_xlabel("Model score / probability")
    ax.set_ylabel("Count")
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)


def render(threshold, prevalence, separability, color_mode):
    y_true, y_score = make_dataset(n=100,
                                   prevalence=float(prevalence),
                                   separability=float(separability))

    y_pred = (y_score >= threshold).astype(int)

    tp, fp, fn, tn = confusion_counts(y_true, y_pred)
    acc, prec, rec, f1 = compute_metrics(tp, fp, fn, tn)

    clear_output(wait=True)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    plot_confusion(axes[0], tp, fp, fn, tn)
    plot_score_hist(axes[1], y_true, y_score, y_pred, threshold, color_mode)
    plt.show()

    n_pos = int(np.sum(y_true == 1))
    n_neg = int(np.sum(y_true == 0))
    print(f"Dataset: n=100  positives={n_pos}  negatives={n_neg}  prevalence={n_pos/100:.2f}")
    print(f"Threshold: {threshold:.2f}")
    print(f"TP={tp}  FP={fp}  FN={fn}  TN={tn}\n")
    print(f"Accuracy   = {fmt(acc)}")
    print(f"Precision  = {fmt(prec)}")
    print(f"Recall/TPR = {fmt(rec)}")
    print(f"F1         = {fmt(f1)}")

thr = widgets.FloatSlider(
    value=0.50, min=0.0, max=1.0, step=0.01,
    description="Threshold",
    continuous_update=True,
    readout_format=".2f",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="650px")
)

prevalence = widgets.FloatSlider(
    value=0.20, min=0.01, max=0.99, step=0.01,
    description="Positive rate (prevalence)",
    continuous_update=False,
    readout_format=".2f",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="650px")
)

separability = widgets.FloatSlider(
    value=2.0, min=0.0, max=5.0, step=0.1,
    description="Separability (hard → easy)",
    continuous_update=False,
    readout_format=".1f",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="650px")
)

color_mode = widgets.ToggleButtons(
    options=["By true label", "By outcome (TP/FP/FN/TN)"],
    value="By true label",
    description="Histogram colors",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="650px")
)

ui = widgets.VBox([thr, prevalence, separability, color_mode])
out = widgets.interactive_output(render, {
    "threshold": thr,
    "prevalence": prevalence,
    "separability": separability,
    "color_mode": color_mode
})

display(ui, out)



Move the sliders above and observe how the confusion matrix and metrics change.

Q. Threshold effects: As you slowly increase the threshold from left to right,

- What trend do you notice in **precision**?
- What trend do you notice in **recall**?
- Explain your observations using TP, FP, and FN.

`YOUR ANSWER HERE`

Q. Extreme thresholds: Set the threshold very close to **0** (almost everything predicted positive).  
  What happens to recall? What happens to precision?

`YOUR ANSWER HERE`

Q. Extreme thresholds: Set the threshold very close to **1** (almost nothing predicted positive).  
  What happens to recall? What happens to precision?

`YOUR ANSWER HERE`


Q. Imbalanced data (prevalence slider). Lower the positive rate so that positives become rare. What happens to **accuracy** compared to precision and recall? Why might accuracy remain high even when recall is low?

`YOUR ANSWER HERE`


Q. Increase separability. What changes do you observe in the confusion matrix?  Does changing the threshold matter as much when separability is high? Why or why not?

`YOUR ANSWER HERE`


Q Switch to the outcome coloring. Where do most **false positives** appear relative to the threshold line?  Where do most **false negatives** appear?

`YOUR ANSWER HERE`

## From confidence to loss

Above we saw how, given a model (a way of calculating the score), we can predict the label of of an input example.

To choose the model parameters for a given training set, we will want a way to measure the loss of a model.

A **loss function** tells us how bad the model’s prediction was.

- Small loss -> the model did well
- Large loss -> the model did poorly

***Note**: We will switch the classes of $y$ from $-1$ and $1$ to $0$ and $1$ for mathematical convenience. This change does not make a difference to the underlying logic or results.*

For binary classification, we want a loss function such that:

- If the true label is 1:
  - high confidence (close to 1) → small loss
  - low confidence (close to 0) → large loss

- If the true label is 0:
  - low confidence (close to 0) → small loss
  - high confidence (close to 1) → large loss

#### Reminders about logarithms

$$log(a) = b \qquad\text{means} \qquad e^b = a$$

In particular:

- $log(1) = 0$
- For $a < 0$, $log(a)$ is undefined.
- For $0 < a < 1$, $log(a) < 0$.
- As $a$ increases, $log(a)$ increases.

In [ ]:
# e^x works for any real x; choose a reasonable range
x_exp = np.linspace(-2, 2, 400)
# log(x) is only defined for x > 0; avoid the singularity at 0
x_log = np.linspace(0.01, 4, 400)

# Compute the function values
y_exp = np.exp(x_exp)          # e^x
y_log = np.log(x_log)          # natural logarithm

# Create side‑by‑side subplots ------------------------
fig, (ax2, ax1) = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)


# Plot e^x
ax1.plot(x_exp, y_exp, color='tab:blue')
ax1.set_title(r'$a = e^b$')
ax1.set_xlabel('b')
ax1.set_ylabel('a')
ax1.set_xticks(range(-2,3))
ax1.grid(True, which='both', linestyle='--', linewidth=0.5)

# Plot log(x)
ax2.plot(x_log, y_log, color='tab:orange')
ax2.set_title(r'$b = \log(a)$')
ax2.set_xlabel('a')
ax2.set_xticks(range(4))
ax2.set_ylabel('b')
ax2.grid(True, which='both', linestyle='--', linewidth=0.5)

plt.show()

### Binary cross-entropy

One commonly used loss function is called **binary cross-entropy**. 

Binary cross-entropy loss for one data point is defined to be:

- For true label value $y=1$ and confidence $p$: $\mathcal{L}(p, 1) = -\log(p)$  
- For true label value $y=0$ and confidence $p$: $\mathcal{L}(p, 0) = -\log(1-p)$


#### Where does this formula come from?

- When the model is **very confident and correct**,  say it says "95% chance of class 1" and the true label really is 1, the penalty should be **small** (nearly 0). Comparing this to the formula

$$\mathcal{L}(0.95, 1) = -\log(0.95) \approx - \log(1) = -0 = 0$$

- When the model is **very confident and wrong**, say it says "95% chance of class 1" but the true label is 0,  the penalty should be **large** (very high). Comparing this to the formula

$$\mathcal{L}(0.95, 0) = -\log(1-0.95) = - \log(0.05) = -\text{large negative number} = \text{large positive number}$$

- The penalty should grow **smoothly** as the model's confidence moves in the wrong direction.


When $p$ is close to 1 (model is confident and correct for $y=1$), $-\log(p)$ is close to 0.  
When $p$ is close to 0 (model is wrong for $y=1$), $-\log(p)$ shoots up toward infinity.

**Why logarithm specifically?**

Two key properties of logarithm that match the intended properties for loss:

1. $\log(1) = 0$ means no penalty when the model is perfectly right.
2. $\log(p) \to -\infty$ as $p \to 0$ so the penalty grows without bound when the model is completely wrong with total confidence.

No other simple function has both properties at once. This is why binary cross-entropy is the standard loss for binary classification.

In [ ]:
p_vals = np.linspace(0.001, 0.999, 500)
bce_y1 = [-np.log(p) for p in p_vals]           # y = 1
bce_y0 = [-np.log(1 - p) for p in p_vals]       # y = 0
plt.figure(figsize=(7, 4))
plt.plot(p_vals, bce_y1, label="y = 1")
#plt.plot(p_vals, bce_y0, label="y = 0")
plt.xlabel("$p$")
plt.ylabel("Binary cross-entropy loss")
plt.title("Binary Cross-Entropy Loss vs Confidence")
plt.legend()
plt.axvline(x=0.5, color='gray', linestyle='--', linewidth=2, alpha=0.5)
plt.grid(True)
plt.show()

We can write the piecewise definition of binary cross-entropy more succinctly by combining the two cases:
$$
\mathcal{L}(p, y) = -\big(y \log(p) + (1-y)\log(1-p)\big)
$$

where:
- $p$ is the model’s confidence (from sigmoid)
- $y$ is the true label (0 or 1)


The two cases (true label = 1 and true label = 0) are combined into one formula so we don't need an if/else:

$$
\mathcal{L}(p, y) = -\big(y \log(p) + (1-y)\log(1-p)\big)
$$

Run the cells below to see why this works.

In [ ]:
# Complete the binary cross-entropy loss function below.

def binary_cross_entropy(p, y):
    """
    p: model confidence [0, 1]
    y: true label (0 or 1)
    """

    # Return the binary cross-entropy loss: -(y * log(p) + (1 - y) * log(1 - p))
    # You can use math.log() to compute the logarithm of a number.

    # YOUR CODE HERE

    # raise NotImplementedError

In [ ]:
# True label = 1
# When y = 1, the (1-y) term equals 0, 
# so only -log(p) remains, and that's the one we wanted in this case.

print("When y=1, want high p to give low loss and low p to give high loss.")
print(f"binary_cross_entropy(0.9, 1) = {binary_cross_entropy(0.9, 1)}")
print(f"binary_cross_entropy(0.1, 1) = {binary_cross_entropy(0.1, 1)}")


# True label = 0
# When y = 0, the y term equals 0, so only -log(1-p) remains, and that's the one we wanted in this case.

print("\nWhen y=0, want high p to give high loss and low p to give low loss.")
print(f"binary_cross_entropy(0.1, 0) = {binary_cross_entropy(0.1, 0)}")
print(f"binary_cross_entropy(0.9, 0) = {binary_cross_entropy(0.1, 0)}")

Q. We calculate binary cross entropy for values of $p$ in the interval $[0,1]$. For which values of $p$ is this loss calculation undefined or numerically unstable? Why?

`YOUR ANSWER HERE`


In [ ]:
def stable_binary_cross_entropy(p, y):
    """
    Clips p to avoid (overflow) issues, then computes binary cross-entropy loss.
    
    p: model confidence (between 0 and 1)
    y: true label (0 or 1)
    """
    eps = 1e-8

    if p < eps:
        p = eps
    if p > 1 - eps:
        p = 1 - eps

    return -(y * math.log(p) + (1 - y) * math.log(1 - p))

In [ ]:
# should not error out
try: 
    print(binary_cross_entropy(0, 1))
except:
    print("Error: binary_cross_entropy(0, 1)")

print(stable_binary_cross_entropy(0, 1))

try: 
    print(binary_cross_entropy(1, 1))
except:
    print("Error: binary_cross_entropy(1, 1)")

print(stable_binary_cross_entropy(1, 1))

try: 
    print(binary_cross_entropy(0, 0))
except:
    print("Error: binary_cross_entropy(0, 0)")
print(stable_binary_cross_entropy(0, 0)) 

try:
    print(binary_cross_entropy(1, 0))
except:
    print("Error: binary_cross_entropy(1, 0)")
print(stable_binary_cross_entropy(1, 0)) 

In [ ]:
# Let's visualize the relationship between model parameters, score, confidence, and loss

# Load the toy data again
X_toy = [
    [1.5, 4],
    [1, 2],
    [2, 1],
    [3, 5],
    [4, 2],
    [0, 0],
    [1.5, -0.5]
]
y_toy = [1, 1, 0, 1, 0, 0, 0]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

def combined_interactive_plot(w1=1.0, w2=1.0, b=0.0):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # LEFT PLOT: Sigmoid Confidence (1D View)
    z_range = np.linspace(-10, 10, 400)
    #sig_vals = 1 / (1 + np.exp(-z_range))
    sig_vals = [stable_sigmoid(z) for z in z_range]

    ax1.plot(z_range, sig_vals, color='green', label="Sigmoid(z)")
    ax1.axvline(x=0, color='black', linestyle='--', alpha=0.5, label="Boundary (z=0)")
    
    # Plot toy points on the sigmoid curve and scatter plot 
    # and store point losses for reuse
    total_loss = 0
    point_data = [] # list of (x,y_label,z_score, p, loss)
    
    #for ax2
    labeled0 = False
    labeled1 = False

    for x, y in zip(X_toy, y_toy):
        # For sigmoid curve plot
        z_score = w1 * x[0] + w2 * x[1] + b
        p = stable_sigmoid(z_score)
        loss = stable_binary_cross_entropy(p,y)
        point_data.append((x,y,z_score,p,loss))
        total_loss += loss
        
        color = 'red' if y == 1 else 'blue'
        ax1.scatter(z_score, p, color=color, s=50, edgecolors='k', zorder=5)
        
        label = f"x1 ={x[0]},x2={x[1]}\nL={loss:.2f}"
        ax1.annotate(
            label,
            xy=(z_score,p),
            xytext=(8,4),
            textcoords="offset points",
            fontsize=7,
            color=color,
            bbox=dict(boxstyle="round,pad=0.2",fc="white",alpha=0.6,ec=color),
            arrowprops=dict(arrowstyle="-",color=color,lw=0.8)
        )
        # For scatter plot
        if y == 1:
            ax2.scatter(x[0], x[1], c='red', marker='x', s=100,
                label="Class 1" if not labeled1 else "")
            labeled1 = True
        else:
            ax2.scatter(x[0], x[1], c='blue', marker='o', s=100,
                label="Class 0" if not labeled0 else "")
            labeled0 = True
        color = 'red' if y == 1 else 'blue'
        ax2.annotate(
            f"L={loss:.2f}",
            xy=(x[0],x[1]),
            xytext=(5,5),
            textcoords="offset points",
            fontsize=8,
            color=color,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7, ec=color),
        )    
    
    mean_loss = total_loss/len(X_toy)
    
    ax1.set_xlabel("Score ($z = w_1x_1 + w_2x_2 + b$)")
    ax1.set_ylabel("Confidence ($p$)")
    ax1.set_title(f"1D View: Confidence vs Score\nMean CE Loss: {mean_loss:.2f}")
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    # RIGHT PLOT: Decision Boundary (2D View)
    x1_min, x1_max = -1, 5
    x2_min, x2_max = -1, 6
    xx, yy = np.meshgrid(np.linspace(x1_min, x1_max, 100), np.linspace(x2_min, x2_max, 100))
    z_grid = w1*xx + w2*yy + b
    stable_sigmoid_vec = np.vectorize(stable_sigmoid)
    p_grid = stable_sigmoid_vec(z_grid)
    # Background probability contour
    contour = ax2.imshow(p_grid, origin="lower", extent=[x1_min, x1_max, x2_min, x2_max], 
                         aspect="auto", alpha=0.3, cmap='RdBu_r')
    
    # Decision boundary line (where z=0)
    if w2 != 0:
        x1_vals = np.array([x1_min, x1_max])
        x2_vals = -(w1 * x1_vals + b) / w2
        ax2.plot(x1_vals, x2_vals, "k--", label="z=0")
    elif w1 != 0:
        x0 = -b / w1
        ax2.axvline(x=x0, color="k", linestyle="--", label="z=0")
    
    ax2.set_xlim(x1_min, x1_max)
    ax2.set_ylim(x2_min, x2_max)
    ax2.set_xlabel("Feature $x_1$")
    ax2.set_ylabel("Feature $x_2$")
    ax2.set_title("2D View: Feature Space Boundary")
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    

interact(
    combined_interactive_plot,
    w1=FloatSlider(value=1.0, min=-5, max=5, step=0.1),
    w2=FloatSlider(value=1.0, min=-5, max=5, step=0.1),
    b=FloatSlider(value=-2.0, min=-5, max=5, step=0.1)
)

What changes in the graph when the loss changes?

`YOUR ANSWER HERE`

Summarizing what we did today: given model parameters $w_i$ and $b$, to predict the label of an input example and evaluate the prediction:

1. Compute a score
   $$z = \sum_{i=1}^n w_i x_i + b$$

2. Turn the score into a confidence using the sigmoid function
   $$p = \sigma(z)$$

3. Measure how good that confidence was using a loss (given the true label $y$)
   $$\mathcal{L}(p, y) = -\big(y \log(p) + (1-y)\log(1-p)\big)$$

We call this the *forward* computation:
  
$$x \xrightarrow{\,w,b\,} z \xrightarrow{\,\sigma(.)\,} p \xrightarrow{\,y\,} \mathcal{L}$$

Each arrow means "is computed from" or "depends on". This sequence is the **computation graph** for our model.



Next time we will ask:

- How should we change the parameters $w$ and $b$ so that the loss $L$ becomes smaller?


$x$ is the data, we can't change it.
$w,b$ are the model parameters.
$z$ is the score computed from the data and the model parameters.
$p$ is the confidence computed from the score.
$\mathcal{L}$ is the loss.

The only thing we can change are the model parameters!